In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
import optuna
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

import warnings
warnings.simplefilter('ignore')

SEED = 30

In [2]:
df_train = pd.read_csv('/kaggle/input/playground-series-s5e7/train.csv')
df_test = pd.read_csv('/kaggle/input/playground-series-s5e7/test.csv')

In [3]:
# Making Data Compatible with XGBoost
outcome_mapping = {
    'Introvert': 0,
    'Extrovert': 1
}

reverse_mapping = {v: k for k, v in outcome_mapping.items()}

def clean_data(df, test=False):
    """
    Function for Cleaning Data
    Takes in dataframe and applies data cleaning:
    - Encodes 'objects' as 'category'
    - Drops the id column
    - if test is equal to False, then also encode the outcome numerically
    """
    df_temp = df.copy()
    cat_features = list(df_temp.select_dtypes(include=['object']).columns)
    df_temp[cat_features] = df_temp[cat_features].astype('category')
    df_temp.drop(columns=['id'], inplace=True)
    if not test:
        df_temp['Personality'] = df_temp['Personality'].map(outcome_mapping)

    return df_temp

In [4]:
df_train_clean = clean_data(df_train)

X = df_train_clean.drop(columns=['Personality'])
y = df_train_clean['Personality']

In [5]:
def cross_val(X, y, params, K=10, debug=False):
    # Cross Validation
    kf = StratifiedKFold(n_splits=K, shuffle=True, random_state=SEED)
    scores = []
    
    for i, (train_idx, val_idx) in enumerate(kf.split(X, y), 1):
        X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
        y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]
    
        model = XGBClassifier(
            **params
        )
        model.fit(X_train_fold,
                  y_train_fold,
                  eval_set=[(X_val_fold, y_val_fold)],
                  verbose=False)
        fold_val_pred = model.predict(X_val_fold)
    
        score = accuracy_score(y_val_fold, fold_val_pred)
        if debug == True:
            print(f'========== Fold {i} accuracy Score: {score} ==========')
        scores.append(score)
    if debug == True:
        print(f'Average Score: {np.mean(scores)}')
    return np.mean(scores)

In [6]:
def objective(trial):
    params = {
        "device": 'cuda',
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, step=0.01),
        "max_depth": trial.suggest_int("max_depth", 3, 20),
        "subsample": trial.suggest_float("subsample", 0.1, 1.0, step=0.1),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.1, 1.0, step=0.1),
        "max_bin": trial.suggest_int("max_bin", 256, 2048),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0, 0.1, step=0.01),
        "lambda": trial.suggest_float("lambda", 1e-3, 10.0, log=True),
        "alpha": trial.suggest_float("alpha", 1e-3, 10.0, log=True),
        "max_delta_step": trial.suggest_int("max_delta_step", 1, 10),
        "n_estimators": 3000,
        "enable_categorical": True,
        "early_stopping_rounds":100,
        "random_state": SEED
    }

    score = cross_val(X, y, params=params, debug=False, K=10)
    return score

In [7]:
%%time
study = optuna.create_study(direction='maximize',
                            sampler = optuna.samplers.RandomSampler(seed=SEED),
                            study_name = "BIG BLUE FIN TUNA!!")
study.optimize(objective, n_trials=150, show_progress_bar=True)

[I 2025-07-03 22:45:01,699] A new study created in memory with name: BIG BLUE FIN TUNA!!


  0%|          | 0/150 [00:00<?, ?it/s]

[I 2025-07-03 22:45:12,329] Trial 0 finished with value: 0.968851136269595 and parameters: {'learning_rate': 0.06999999999999999, 'max_depth': 9, 'subsample': 0.7000000000000001, 'colsample_bytree': 0.2, 'max_bin': 1981, 'min_child_weight': 4, 'gamma': 0.1, 'lambda': 0.008714281448909188, 'alpha': 0.22017959781029373, 'max_delta_step': 5}. Best is trial 0 with value: 0.968851136269595.
[I 2025-07-03 22:45:23,263] Trial 1 finished with value: 0.9691210855317219 and parameters: {'learning_rate': 0.02, 'max_depth': 12, 'subsample': 0.6, 'colsample_bytree': 0.8, 'max_bin': 1930, 'min_child_weight': 1, 'gamma': 0.02, 'lambda': 9.479266587302769, 'alpha': 0.008724127227938479, 'max_delta_step': 3}. Best is trial 1 with value: 0.9691210855317219.
[I 2025-07-03 22:45:32,259] Trial 2 finished with value: 0.968851136269595 and parameters: {'learning_rate': 0.06999999999999999, 'max_depth': 16, 'subsample': 0.7000000000000001, 'colsample_bytree': 0.1, 'max_bin': 1874, 'min_child_weight': 3, 'gamm

In [8]:
best_params = study.best_params
print(f'Best Trial Params: {best_params}')

print(f'Best Trial Value: {study.best_trial.value}')

Best Trial Params: {'learning_rate': 0.09, 'max_depth': 19, 'subsample': 0.4, 'colsample_bytree': 0.9, 'max_bin': 1858, 'min_child_weight': 10, 'gamma': 0.0, 'lambda': 0.003263776283246641, 'alpha': 0.0011515717758325298, 'max_delta_step': 7}
Best Trial Value: 0.9694450013345938


In [9]:
best_params["n_estimators"] =  3000
best_params["enable_categorical"] = True
# best_params["early_stopping_rounds"] = 100
best_params["random_state"] = SEED
best_params["device"] = 'cuda'

In [10]:
print('========== XGB BEST PARAMS CROSS VALIDATION ==========')
avg_score = cross_val(X, y, params=best_params, debug=True, K=10)

========== XGB BEST PARAMS CROSS VALIDATION ==========
========== Fold 1 accuracy Score: 0.9665407447382622 ==========
========== Fold 2 accuracy Score: 0.9767943874797625 ==========
========== Fold 3 accuracy Score: 0.9638424177010254 ==========
========== Fold 4 accuracy Score: 0.9643820831084727 ==========
========== Fold 5 accuracy Score: 0.9638228941684666 ==========
========== Fold 6 accuracy Score: 0.964902807775378 ==========
========== Fold 7 accuracy Score: 0.9697624190064795 ==========
========== Fold 8 accuracy Score: 0.9681425485961123 ==========
========== Fold 9 accuracy Score: 0.9681425485961123 ==========
========== Fold 10 accuracy Score: 0.9719222462203023 ==========
Average Score: 0.9678255097390374


In [11]:
df_test_clean = clean_data(df_test, test=True)
model = XGBClassifier(
    **best_params
)

model.fit(X, y)
y_test_pred = model.predict(df_test_clean)

In [12]:
submission = pd.read_csv('/kaggle/input/playground-series-s5e7/sample_submission.csv')
submission['Personality'] = y_test_pred
submission['Personality'] = submission['Personality'].map(reverse_mapping)
submission.to_csv('submission.csv', index=False)
submission.head()

,id,Personality
0,18524,Extrovert
1,18525,Introvert
2,18526,Extrovert
3,18527,Extrovert
4,18528,Introvert
